In [1]:
import os, random
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup


In [2]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)


In [3]:
DATA_PATH = r"C:/Users/fatem/Downloads/spam.csv"
LABEL_RAW_COL = "v1"
TEXT_RAW_COL  = "v2"

MODEL_NAME = "distilroberta-base"  
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 2
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
CLIP_NORM = 1.0
GRAD_ACCUM_STEPS = 1
VAL_SIZE = 0.1

CAP_TRAIN_PER_CLASS = 2000  
CAP_VAL_PER_CLASS   = 500   

SAVE_DIR = "english_fraud_best"


In [4]:
df = pd.read_csv(DATA_PATH, encoding="latin-1")

df = df.rename(columns={LABEL_RAW_COL: "label_str", TEXT_RAW_COL: "text"})
df = df[["label_str", "text"]].dropna()
df["text"] = df["text"].astype(str)

df["label"] = df["label_str"].map({"ham": 0, "spam": 1})
df = df.dropna(subset=["label"])
df["label"] = df["label"].astype(int)

NUM_LABELS = df["label"].nunique()
print("NUM_LABELS:", NUM_LABELS)
print("Data shape:", df.shape)
print("Label distribution:\n", df["label"].value_counts().sort_index())


NUM_LABELS: 2
Data shape: (5572, 3)
Label distribution:
 label
0    4825
1     747
Name: count, dtype: int64


In [5]:
tr_df, val_df = train_test_split(
    df,
    test_size=VAL_SIZE,
    random_state=42,
    stratify=df["label"]
)

print("\nSplit:")
print("Train:", tr_df.shape, "Val:", val_df.shape)



Split:
Train: (5014, 3) Val: (558, 3)


In [6]:
def stratified_cap(df_in, label_col, cap_per_class, seed=42):
    if cap_per_class is None:
        return df_in.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    parts = []
    for lab, grp in df_in.groupby(label_col):
        if len(grp) > cap_per_class:
            grp = grp.sample(n=cap_per_class, random_state=seed)
        parts.append(grp)
    out = pd.concat(parts).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return out

tr_small = stratified_cap(tr_df, "label", CAP_TRAIN_PER_CLASS, seed=42)
val_small = stratified_cap(val_df, "label", CAP_VAL_PER_CLASS, seed=42)

print("\nAfter capping (if enabled):")
print("Train:", tr_small.shape, "Val:", val_small.shape)
print("Train label dist:\n", tr_small["label"].value_counts().sort_index())



After capping (if enabled):
Train: (2672, 3) Val: (558, 3)
Train label dist:
 label
0    2000
1     672
Name: count, dtype: int64


In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class SpamDataset(Dataset):
    def __init__(self, df_in):
        self.texts = df_in["text"].tolist()
        self.labels = df_in["label"].tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

C:\Users\fatem\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\fatem\.cache\huggingface\hub\models--distilroberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [8]:
train_ds = SpamDataset(tr_small)
val_ds   = SpamDataset(val_small)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("\nBatches:", len(train_loader), len(val_loader))



Batches: 167 35


In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS
).to(device)


Using device: cpu


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   3%|3         | 10.5M/331M [00:00<?, ?B/s]

Error while downloading from https://huggingface.co/distilroberta-base/resolve/main/model.safetensors: HTTPSConnectionPool(host='cas-bridge.xethub.hf.co', port=443): Read timed out.
Trying to resume download...


model.safetensors:  22%|##2       | 73.4M/331M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at distilroberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [11]:
y_train = tr_small["label"].values
classes_present = np.array(sorted(np.unique(y_train)))
weights = compute_class_weight(class_weight="balanced", classes=classes_present, y=y_train)

full_w = np.ones(NUM_LABELS, dtype=np.float32)
for c, w in zip(classes_present, weights):
    full_w[int(c)] = w

class_weights = torch.tensor(full_w, dtype=torch.float).to(device)
loss_fn = CrossEntropyLoss(weight=class_weights)

print("Class weights:", class_weights.detach().cpu().numpy())


Class weights: [0.668     1.9880953]


In [12]:
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

total_steps = (len(train_loader) // GRAD_ACCUM_STEPS) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)


In [13]:
def train_one_epoch(epoch):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    loop = tqdm(train_loader, desc=f"Train Epoch {epoch}")
    for step, batch in enumerate(loop, start=1):
        input_ids = batch["input_ids"].to(device)
        attn_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        out = model(input_ids=input_ids, attention_mask=attn_mask)
        logits = out.logits

        loss = loss_fn(logits, labels) / GRAD_ACCUM_STEPS
        loss.backward()

        if step % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        total_loss += loss.item() * GRAD_ACCUM_STEPS
        loop.set_postfix(loss=f"{(total_loss/step):.4f}")

    return total_loss / len(train_loader)


In [14]:
@torch.no_grad()
def evaluate(loader, title="Val"):
    model.eval()
    all_preds, all_labels = [], []

    for batch in tqdm(loader, desc=f"Eval {title}"):
        input_ids = batch["input_ids"].to(device)
        attn_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        out = model(input_ids=input_ids, attention_mask=attn_mask)
        preds = torch.argmax(out.logits, dim=1)

        all_preds.extend(preds.cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())

    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    weighted_f1 = f1_score(all_labels, all_preds, average="weighted")

    print(f"\n{title} Accuracy   : {acc:.4f}")
    print(f"{title} Macro F1   : {macro_f1:.4f}")
    print(f"{title} Weighted F1: {weighted_f1:.4f}")
    print("\nClassification report:")
    print(classification_report(all_labels, all_preds, digits=4))
    print("Confusion matrix:")
    print(confusion_matrix(all_labels, all_preds))

    return macro_f1


In [15]:
best_macro = -1.0
os.makedirs(SAVE_DIR, exist_ok=True)

for epoch in range(1, EPOCHS + 1):
    avg_loss = train_one_epoch(epoch)
    print(f"\nEpoch {epoch} avg train loss: {avg_loss:.4f}")

    macro = evaluate(val_loader, title="Val")

    if macro > best_macro:
        best_macro = macro
        model.save_pretrained(SAVE_DIR)
        tokenizer.save_pretrained(SAVE_DIR)
        print(f"\n✅ Saved best model to: {SAVE_DIR} (best macro-F1={best_macro:.4f})")


Train Epoch 1: 100%|████████████████████████████████████████████████████| 167/167 [15:44<00:00,  5.66s/it, loss=0.2089]



Epoch 1 avg train loss: 0.2089


Eval Val: 100%|████████████████████████████████████████████████████████████████████████| 35/35 [00:42<00:00,  1.20s/it]



Val Accuracy   : 0.9910
Val Macro F1   : 0.9802
Val Weighted F1: 0.9909

Classification report:
              precision    recall  f1-score   support

           0     0.9898    1.0000    0.9949       483
           1     1.0000    0.9333    0.9655        75

    accuracy                         0.9910       558
   macro avg     0.9949    0.9667    0.9802       558
weighted avg     0.9911    0.9910    0.9909       558

Confusion matrix:
[[483   0]
 [  5  70]]

✅ Saved best model to: english_fraud_best (best macro-F1=0.9802)


Train Epoch 2: 100%|████████████████████████████████████████████████████| 167/167 [17:46<00:00,  6.39s/it, loss=0.0438]



Epoch 2 avg train loss: 0.0438


Eval Val: 100%|████████████████████████████████████████████████████████████████████████| 35/35 [00:42<00:00,  1.23s/it]


Val Accuracy   : 0.9857
Val Macro F1   : 0.9702
Val Weighted F1: 0.9859

Classification report:
              precision    recall  f1-score   support

           0     0.9979    0.9855    0.9917       483
           1     0.9136    0.9867    0.9487        75

    accuracy                         0.9857       558
   macro avg     0.9557    0.9861    0.9702       558
weighted avg     0.9866    0.9857    0.9859       558

Confusion matrix:
[[476   7]
 [  1  74]]


In [20]:
@torch.no_grad()
def predict_text(text: str):
    model.eval()
    enc = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=MAX_LEN)
    enc = {k: v.to(device) for k, v in enc.items()}
    out = model(**enc)
    return torch.argmax(out.logits, dim=1).item()

print("\nManual tests:")
samples = [
    "Hey, are we still meeting tomorrow?",
    "URGENT! You have won a free prize. Click the link to claim now!",
    "Call me when you get home.",
    "Congratulations! You were selected for a cash reward. Text WIN to 87121.",
        "Hey, are we still meeting tomorrow at 6?",
    "URGENT! You won a free iPhone. Click this link to claim now!",
    "Your bank account is locked. Verify your details immediately.",
    "I will call you later, I'm in a meeting.",
     "Hey, are we still meeting tomorrow at 6?",
    "I’m on my way, see you in 10 minutes.",
    "Can you send me the notes from today’s class?",
    "Don’t forget to bring your ID card for the appointment.",
    "Happy birthday! Hope you have an amazing day!",
    "Thanks for your help today, I really appreciate it.",
    "Let’s have lunch after work if you’re free.",
    "Call me when you finish your meeting.",
    "I’ll be late, traffic is terrible right now.",
    "Did you talk to Sarah about the project?",
    "Please email me the final PDF when you’re done.",
    "I’m at the supermarket—do you need anything?",
    "See you at the library around 4 pm.",
    "Good night. Talk to you tomorrow.",
    "I’m feeling sick today, I might stay home.",
    "Can you pick me up from the station?",
    "The lecture was moved to 2 pm, don’t be late.",
    "I’ve sent the payment already. Did you receive it?",
    "Let me know when you’re free this weekend.",
    "Your package has been delivered to the front desk.",
     "URGENT! You won a free iPhone. Click this link to claim now!",
    "Congratulations! You have been selected to receive a $1,000 gift card. Reply YES to claim.",
    "Your bank account is locked. Verify your details immediately: http://secure-login-verify.com",
    "FINAL NOTICE: Unpaid bill detected. Pay now to avoid legal action: http://pay-now-support.com",
    "You have a pending refund of $245. Confirm your account information to receive it.",
    "WINNER!! Claim your prize now. Text WIN to 87121.",
    "Your PayPal has been limited. Restore access by confirming here: http://paypal-confirmation.com",
    "You are eligible for a loan approval. No credit check. Apply now!",
    "We tried to deliver your parcel. Schedule redelivery here: http://parcel-track-update.com",
    "Get rich fast! Earn $5000/week working from home. Click to start.",
    "Your Netflix subscription will be canceled. Update payment details now.",
    "Claim your FREE vacation to Bahamas! Pay small fee to confirm your booking.",
    "Security alert: suspicious login detected. Reset your password immediately via this link.",
    "You have been overcharged. Tap here to claim compensation now!",
    "Limited offer! Buy 1 get 1 free. Click here before it expires!",
    "Your Apple ID has been suspended. Verify immediately to restore access.",
    "Exclusive deal: 90% discount on electronics. Shop now: http://super-sale-now.com",
    "Urgent: IRS tax refund waiting. Confirm your SSN to receive it.",
    "Your card has been flagged for unusual activity. Confirm details to unblock it.",
    "Act now! You have 24 hours to claim your reward.",
        "Hi, this is your courier. I couldn’t reach you—can you confirm your address?",
    "Your account needs verification. Please reply with your full name and date of birth.",
    "Reminder: Your payment is due today. Reply if you need help.",
    "Hey, I found a great deal for you—click here and check it out!",
    "We noticed unusual activity. Please confirm if this was you.",
    "Your OTP is 482931. Do not share it with anyone.",
    "Your order is delayed. Contact support if you have questions.",
    "Hi mom, my phone broke. Save this number and send me money ASAP.",
    "You have a missed call. Call back to avoid charges.",
    "This is a test message. Please ignore."
]
for s in samples:
    pred = predict_text(s)
    print(s, "=>", pred, "(0=ham, 1=spam)")



Manual tests:
Hey, are we still meeting tomorrow? => 0 (0=ham, 1=spam)
URGENT! You have won a free prize. Click the link to claim now! => 1 (0=ham, 1=spam)
Call me when you get home. => 0 (0=ham, 1=spam)
Congratulations! You were selected for a cash reward. Text WIN to 87121. => 1 (0=ham, 1=spam)
Hey, are we still meeting tomorrow at 6? => 0 (0=ham, 1=spam)
URGENT! You won a free iPhone. Click this link to claim now! => 1 (0=ham, 1=spam)
Your bank account is locked. Verify your details immediately. => 0 (0=ham, 1=spam)
I will call you later, I'm in a meeting. => 0 (0=ham, 1=spam)
Hey, are we still meeting tomorrow at 6? => 0 (0=ham, 1=spam)
I’m on my way, see you in 10 minutes. => 0 (0=ham, 1=spam)
Can you send me the notes from today’s class? => 0 (0=ham, 1=spam)
Don’t forget to bring your ID card for the appointment. => 0 (0=ham, 1=spam)
Happy birthday! Hope you have an amazing day! => 0 (0=ham, 1=spam)
Thanks for your help today, I really appreciate it. => 0 (0=ham, 1=spam)
Let’s h